# Mixtral-8x7B Baseline Comparison

**Purpose:** Compare MoE-PolicyLang against Fiddler and MoE-Infinity on Mixtral-8x7B.

This notebook:
1. Runs Fiddler (CPU-GPU hybrid, popularity-based placement)
2. Runs MoE-Infinity (activation-aware offloading, sequence-level tracing)
3. Runs MoE-PolicyLang with equivalent DSL policies
4. Measures throughput (tok/s), cache hit rate, and VRAM usage
5. Produces comparison figures for the paper

**Hardware:** GPU with >=16 GB VRAM + >=64 GB system RAM

**Statistical requirements:** n>=5 runs, bootstrap 95% CIs (per revision plan)

In [ ]:
# --- Colab setup ---
# Mixtral-8x7B fp16 is ~97 GB — too large for Colab free tier (108 GB disk).
# We use the GPTQ 4-bit quantized version (~25 GB) which fits on local disk.
# All systems load the same quantized weights, keeping comparisons fair.

import os, shutil

IN_COLAB = 'COLAB_GPU' in os.environ or os.path.exists('/content')

if IN_COLAB:
    # Mount GDrive for saving results (not model weights)
    from google.colab import drive
    drive.mount('/content/drive')
    RESULTS_DIR = '/content/drive/MyDrive/MoE-PolicyLang/results'
    OFFLOAD_DIR = '/tmp/offload'
    os.makedirs(RESULTS_DIR, exist_ok=True)
    os.makedirs(OFFLOAD_DIR, exist_ok=True)

    local_free = shutil.disk_usage('/').free / 1e9
    print(f'✓ Local disk free: {local_free:.1f} GB (need ~30 GB for GPTQ model)')
else:
    RESULTS_DIR = None
    OFFLOAD_DIR = '/tmp/moe_offload'
    print('Not on Colab — using local disk')

In [ ]:
# --- Download AWQ quantized Mixtral (~24 GB) to local disk ---
# This fits on Colab's 108 GB local disk with room to spare.
#
# Why AWQ + gptqmodel and not auto-gptq:
#   - auto-gptq is unmaintained and fails to build on current Colab (Py 3.12, Torch 2.5+).
#   - Recent transformers consolidated GPTQ/AWQ behind the gptqmodel backend, so loading
#     any AWQ checkpoint now requires `pip install gptqmodel` (not autoawq).
#   - gptqmodel ships prebuilt wheels for Py 3.10-3.12 / CUDA 12.x, which matches Colab.

if IN_COLAB:
    import subprocess, sys
    try:
        import gptqmodel  # noqa: F401
    except ImportError:
        subprocess.run(
            [sys.executable, '-m', 'pip', 'install', '-q', 'gptqmodel', 'optimum'],
            check=True,
        )

# Use AWQ 4-bit on Colab, fp16 locally (if you have 97 GB disk)
if IN_COLAB:
    MODEL_NAME = 'TheBloke/Mixtral-8x7B-Instruct-v0.1-AWQ'
    MODEL_DIR = None  # use default HF cache on local disk
    DTYPE = None  # AWQ handles its own dtype
    print(f'Using AWQ quantized model: {MODEL_NAME} (~24 GB)')
else:
    MODEL_NAME = 'mistralai/Mixtral-8x7B-Instruct-v0.1'
    MODEL_DIR = None
    print(f'Using fp16 model: {MODEL_NAME}')

print(f'  Free disk: {shutil.disk_usage("/").free / 1e9:.1f} GB')

In [ ]:
import torch
import time
import json
import os
import sys
import gc
import numpy as np
from pathlib import Path

print(f"GPU: {torch.cuda.get_device_name(0)}")
props = torch.cuda.get_device_properties(0)
print(f"VRAM: {props.total_memory / 1e9:.1f} GB")
print(f"PyTorch: {torch.__version__}, CUDA: {torch.version.cuda}")

PROJECT_ROOT = Path(os.getcwd()).parent
TRACES_DIR = PROJECT_ROOT / 'traces'
print(f"Project root: {PROJECT_ROOT}")

In [ ]:
# === Configuration ===
# MODEL_NAME set in cell 2 (GPTQ on Colab, fp16 locally)
MAX_NEW_TOKENS = 128
N_WARMUP = 1
N_RUNS = 5
if not IN_COLAB:
    DTYPE = torch.float16

TEST_PROMPTS = [
    'Explain the difference between transformers and RNNs in three paragraphs.',
    'Write a Python function to implement a priority queue using a heap.',
    'What are the main challenges in deploying large language models on consumer hardware?',
    'Describe the Mixture-of-Experts architecture and its advantages for scaling.',
    'How does expert caching improve MoE inference on memory-constrained GPUs?',
]

results = {}  # system -> {tok_s_per_run, mean_tok_s, std_tok_s, hit_rate, vram_gb}
print(f'Model: {MODEL_NAME}')
print(f'Prompts: {len(TEST_PROMPTS)}, tokens: {MAX_NEW_TOKENS}, runs: {N_RUNS}')

In [ ]:
def benchmark_generation(generate_fn, prompts, n_warmup, n_runs):
    """Generic benchmark wrapper. generate_fn(prompt) -> (n_tokens, elapsed_s, hit_rate_or_None)."""
    all_tok_s = []
    all_hit_rates = []

    for run_idx in range(n_warmup + n_runs):
        run_tokens = 0
        run_time = 0.0
        run_hits = []

        for prompt in prompts:
            n_tokens, elapsed, hr = generate_fn(prompt)
            run_tokens += n_tokens
            run_time += elapsed
            if hr is not None:
                run_hits.append(hr)

        tok_s = run_tokens / run_time if run_time > 0 else 0
        is_warmup = run_idx < n_warmup
        label = 'WARMUP' if is_warmup else f'Run {run_idx - n_warmup + 1}'
        print(f'  {label}: {run_tokens} tokens in {run_time:.2f}s = {tok_s:.2f} tok/s')

        if not is_warmup:
            all_tok_s.append(tok_s)
            if run_hits:
                all_hit_rates.append(np.mean(run_hits))

    vram_gb = torch.cuda.max_memory_allocated() / 1e9
    result = {
        'tok_s_per_run': all_tok_s,
        'mean_tok_s': float(np.mean(all_tok_s)),
        'std_tok_s': float(np.std(all_tok_s)),
        'vram_gb': round(vram_gb, 1),
        'n_runs': n_runs,
    }
    if all_hit_rates:
        result['hit_rate'] = float(np.mean(all_hit_rates))
    return result


def cleanup():
    """Free GPU memory between benchmarks."""
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    print(f'  GPU memory after cleanup: {torch.cuda.memory_allocated() / 1e9:.2f} GB')

---
## 1. Baseline: HuggingFace `device_map="auto"`

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# Baseline = naive HF device_map='auto'. On Mixtral GPTQ (~24 GB) with a 16 GB
# T4, ~8-10 GB spills to CPU and decode runs at ~0.2 tok/s. We use a tiny
# workload: the baseline is a reference point, not the scientific comparison
# (Fiddler / MoE-Infinity / MoE-PolicyLang below are the meaningful results).
BASELINE_N_PROMPTS = 1
BASELINE_MAX_NEW_TOKENS = 32
BASELINE_N_WARMUP = 0
BASELINE_N_RUNS = 2

print('=== Baseline: device_map="auto" ===')
print(f'  Loading: {MODEL_NAME}')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Newer transformers prefers `dtype=` over `torch_dtype=`.
if IN_COLAB:
    model_baseline = AutoModelForCausalLM.from_pretrained(MODEL_NAME, device_map='auto')
else:
    model_baseline = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, dtype=DTYPE, device_map='auto'
    )

def gen_baseline(prompt):
    enc = tokenizer(prompt, return_tensors='pt', padding=True)
    input_ids = enc['input_ids'].to('cuda')
    attention_mask = enc['attention_mask'].to('cuda')
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        out = model_baseline.generate(
            input_ids,
            attention_mask=attention_mask,
            max_new_tokens=BASELINE_MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0
    n_gen = out.shape[1] - input_ids.shape[1]
    print(f'    prompt done: {n_gen} tok in {elapsed:.1f}s = {n_gen/elapsed:.2f} tok/s')
    return n_gen, elapsed, None

baseline_prompts = TEST_PROMPTS[:BASELINE_N_PROMPTS]
print(f'  Baseline workload (reduced): {BASELINE_N_PROMPTS} prompt x '
      f'{BASELINE_MAX_NEW_TOKENS} tok x {BASELINE_N_RUNS} runs')
print(f'  Expected ~5-10 min at ~0.2 tok/s on T4 with CPU offload')

results['baseline_auto'] = benchmark_generation(
    gen_baseline, baseline_prompts, BASELINE_N_WARMUP, BASELINE_N_RUNS
)
print(f'\n  Result: {results["baseline_auto"]["mean_tok_s"]:.2f} +/- '
      f'{results["baseline_auto"]["std_tok_s"]:.2f} tok/s '
      f'(reference only — naive HF offload is intentionally slow here)')

del model_baseline
cleanup()

---
## 2. Fiddler (ICLR 2025)

CPU-GPU hybrid execution with static popularity-based expert placement.

Source: https://github.com/efeslab/fiddler (~280 LOC expert management)

In [ ]:
import subprocess, importlib

fiddler_base = OFFLOAD_DIR if IN_COLAB else '/tmp'
fiddler_path = Path(f'{fiddler_base}/fiddler')
if not fiddler_path.exists():
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/efeslab/fiddler.git', str(fiddler_path)],
                   check=True, capture_output=True)
    print('Cloned Fiddler')
else:
    print('Fiddler already cloned')

sys.path.insert(0, str(fiddler_path / 'src' / 'fiddler'))

In [ ]:
import argparse

# Note: Fiddler may not support GPTQ models directly.
# If it fails, we skip and note in results.
try:
    from mixtral import FiddlerMixtral

    print('=== Fiddler ===')
    fiddler_args = argparse.Namespace(
        model=MODEL_NAME,
        cpu_offload=1,
        beam_width=1,
    )

    fiddler_model = FiddlerMixtral(fiddler_args)

    def gen_fiddler(prompt):
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        prefill_time, decode_time, hit_rate = fiddler_model.generate(
            prompt, output_token=MAX_NEW_TOKENS
        )
        torch.cuda.synchronize()
        elapsed = time.perf_counter() - t0
        return MAX_NEW_TOKENS, elapsed, hit_rate

    results['fiddler'] = benchmark_generation(gen_fiddler, TEST_PROMPTS, N_WARMUP, N_RUNS)
    print(f'\n  Result: {results["fiddler"]["mean_tok_s"]:.2f} +/- {results["fiddler"]["std_tok_s"]:.2f} tok/s')
    print(f'  Hit rate: {results["fiddler"].get("hit_rate", 0):.3f}')

    del fiddler_model
    cleanup()

except Exception as e:
    print(f'⚠ Fiddler failed (may not support GPTQ): {e}')
    results['fiddler'] = {'mean_tok_s': None, 'std_tok_s': None, 'vram_gb': None,
                          'n_runs': 0, 'error': str(e)}

---
## 3. MoE-Infinity

Activation-aware expert offloading with sequence-level tracing.

Source: https://github.com/EfficientMoE/MoE-Infinity (~520 LOC expert management)

In [ ]:
# pip install moe-infinity  (run if not installed)
try:
    import moe_infinity
    print(f'MoE-Infinity version: {moe_infinity.__version__}')
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'moe-infinity'], check=True)
    import moe_infinity

In [ ]:
from moe_infinity import MoE

# Note: MoE-Infinity may not support GPTQ models.
try:
    print('=== MoE-Infinity ===')
    moe_config = {
        'offload_path': OFFLOAD_DIR,
        'device_memory_ratio': 0.75,
    }

    model_moeinf = MoE(MODEL_NAME, moe_config)
    tokenizer_moeinf = AutoTokenizer.from_pretrained(MODEL_NAME)
    if tokenizer_moeinf.pad_token is None:
        tokenizer_moeinf.pad_token = tokenizer_moeinf.eos_token

    def gen_moeinf(prompt):
        inputs = tokenizer_moeinf(prompt, return_tensors='pt').to('cuda')
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        with torch.no_grad():
            out = model_moeinf.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
        torch.cuda.synchronize()
        elapsed = time.perf_counter() - t0
        n_gen = out.shape[1] - inputs['input_ids'].shape[1]
        return n_gen, elapsed, None

    results['moe_infinity'] = benchmark_generation(gen_moeinf, TEST_PROMPTS, N_WARMUP, N_RUNS)
    print(f'\n  Result: {results["moe_infinity"]["mean_tok_s"]:.2f} +/- {results["moe_infinity"]["std_tok_s"]:.2f} tok/s')

    del model_moeinf
    cleanup()

except Exception as e:
    print(f'⚠ MoE-Infinity failed (may not support GPTQ): {e}')
    results['moe_infinity'] = {'mean_tok_s': None, 'std_tok_s': None, 'vram_gb': None,
                               'n_runs': 0, 'error': str(e)}

---
## 4. MoE-PolicyLang (Ours)

Expert-aware loading with DSL-managed caching/prefetch/scheduling.

In [ ]:
sys.path.insert(0, str(PROJECT_ROOT))
from moe_policylang.integrations.loading import load_moe_model
from moe_policylang.integrations import attach

print('=== MoE-PolicyLang ===')
print(f'  Loading: {MODEL_NAME}')

# load_moe_model handles both fp16 and GPTQ models
model_mpl, tokenizer_mpl = load_moe_model(
    MODEL_NAME, torch_dtype=DTYPE if not IN_COLAB else None, gpu_device=0,
)
print(f'  Skeleton VRAM: {torch.cuda.memory_allocated() / 1e9:.1f} GB')

# Fiddler-equivalent: LRU + hybrid scheduling
policy_dsl = '''
policy fiddler_equiv {
    cache { capacity = 8  eviction = lru }
    schedule { mode = hybrid  cpu_threshold_ms = 40.0 }
}
'''
mgr = attach(model_mpl, policy_dsl, gpu_device=0)
print('  Policy: fiddler_equiv (LRU, cap=8, hybrid scheduling)')

def gen_mpl(prompt):
    inputs = tokenizer_mpl(prompt, return_tensors='pt')
    input_ids = inputs['input_ids'].to('cuda')
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        out = model_mpl.generate(input_ids, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0
    n_gen = out.shape[1] - input_ids.shape[1]
    stats = mgr.get_stats()
    hr = stats.get('policy', {}).get('cache', {}).get('hit_rate', None)
    return n_gen, elapsed, hr

results['moe_policylang'] = benchmark_generation(gen_mpl, TEST_PROMPTS, N_WARMUP, N_RUNS)
print(f'\n  Result: {results["moe_policylang"]["mean_tok_s"]:.2f} +/- {results["moe_policylang"]["std_tok_s"]:.2f} tok/s')
print(f'  Hit rate: {results["moe_policylang"].get("hit_rate", "N/A")}')

del model_mpl, mgr
cleanup()

---
## 5. Results Summary & Statistical Analysis

In [ ]:
from scipy import stats as sp_stats

print('=' * 70)
print('RESULTS SUMMARY: Mixtral-8x7B Throughput Comparison')
print('=' * 70)
print(f'{"System":<20} {"tok/s":>14} {"VRAM":>8} {"Hit Rate":>10} {"n":>4}')
print('-' * 60)

for name, r in results.items():
    hr = f'{r["hit_rate"]:.3f}' if 'hit_rate' in r else '---'
    print(f'{name:<20} {r["mean_tok_s"]:>6.2f}+/-{r["std_tok_s"]:.2f} {r["vram_gb"]:>6.1f}GB {hr:>10} {r["n_runs"]:>4}')

print('\n--- Bootstrap 95% CIs (10,000 resamples) ---')
for name, r in results.items():
    data = np.array(r['tok_s_per_run'])
    if len(data) >= 3:
        boot = sp_stats.bootstrap((data,), np.mean, n_resamples=10000, confidence_level=0.95)
        ci_lo, ci_hi = boot.confidence_interval
        print(f'  {name:<20} [{ci_lo:.2f}, {ci_hi:.2f}]')
    else:
        print(f'  {name:<20} (insufficient samples)')

# Save results
output_path = TRACES_DIR / 'mixtral_baseline_comparison.json'
with open(output_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f'\nResults saved to {output_path}')

---
## 6. Comparison Figure

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))

systems = list(results.keys())
means = [results[s]['mean_tok_s'] for s in systems]
stds = [results[s]['std_tok_s'] for s in systems]
vrams = [results[s]['vram_gb'] for s in systems]
labels = ['Baseline\n(auto)', 'Fiddler\n(ICLR\'25)', 'MoE-Infinity', 'MoE-PolicyLang\n(Ours)']
colors = ['#C44E52', '#4C72B0', '#55A868', '#8172B2']

# Throughput bar chart
bars = ax1.bar(range(len(systems)), means, yerr=stds, capsize=5,
               color=colors[:len(systems)], alpha=0.85, edgecolor='white', linewidth=0.8)
ax1.set_ylabel('Throughput (tok/s)', fontsize=11)
ax1.set_title('Mixtral-8x7B Decode Throughput', fontsize=12)
ax1.set_xticks(range(len(systems)))
ax1.set_xticklabels(labels[:len(systems)], fontsize=9)
for bar, val in zip(bars, means):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f'{val:.1f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# VRAM usage
bars2 = ax2.bar(range(len(systems)), vrams,
                color=colors[:len(systems)], alpha=0.85, edgecolor='white', linewidth=0.8)
ax2.set_ylabel('Peak VRAM (GB)', fontsize=11)
ax2.set_title('GPU Memory Usage', fontsize=12)
ax2.set_xticks(range(len(systems)))
ax2.set_xticklabels(labels[:len(systems)], fontsize=9)
for bar, val in zip(bars2, vrams):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
             f'{val:.1f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

fig.suptitle('Mixtral-8x7B: System Comparison (matched hardware, n=5)', fontsize=13, y=1.02)
fig.tight_layout()

fig_dir = PROJECT_ROOT / 'MoE-PolicyLang-Paper' / 'figures'
fig_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(fig_dir / 'mixtral_baseline_comparison.pdf', dpi=150, bbox_inches='tight')
plt.show()
print(f'Figure saved to {fig_dir / "mixtral_baseline_comparison.pdf"}')

---
## 7. Offline Trace-Replay Comparison

In addition to live inference, compare cache policies on recorded Mixtral traces
using MoE-PolicyLang's benchmark harness (no GPU required for this cell).

In [ ]:
from moe_policylang.benchmark.harness import BenchmarkHarness
from moe_policylang.benchmark.policies import get_dsl_policies, BASELINES
from moe_policylang.benchmark.workloads import load_trace_workload

# Load the recorded Mixtral trace
trace_path = TRACES_DIR / 'mixtral_sample.jsonl'
workload = load_trace_workload(trace_path)
print(f'Loaded trace: {workload.name} ({workload.num_tokens} tokens, {workload.num_layers} layers, {workload.num_experts} experts)')

harness = BenchmarkHarness(expert_size_gb=1.2)

# Run all DSL policies
dsl_policies = get_dsl_policies()
offline_results = {}

for name, compiled in dsl_policies.items():
    result = harness.run_policy(compiled, workload)
    m = result.metrics
    offline_results[name] = {
        'hit_rate': m.hit_rate,
        'evictions': m.evictions,
        'us_per_layer': m.mean_dispatch_us,
        'overhead_pct': m.overhead_pct,
    }
    print(f'  {name:<20} hit={m.hit_rate:.4f} evict={m.evictions:>4} lat={m.mean_dispatch_us:.1f}us overhead={m.overhead_pct:.2f}%')

# Run baselines
for bname, (factory, desc) in BASELINES.items():
    result = harness.run_baseline(factory, workload, capacity=16, baseline_name=bname)
    m = result.metrics
    offline_results[bname] = {
        'hit_rate': m.hit_rate,
        'evictions': m.evictions,
        'us_per_layer': m.mean_dispatch_us,
        'overhead_pct': m.overhead_pct,
    }
    print(f'  {bname:<20} hit={m.hit_rate:.4f} evict={m.evictions:>4} lat={m.mean_dispatch_us:.1f}us overhead={m.overhead_pct:.2f}%')

# Save offline results
offline_path = TRACES_DIR / 'mixtral_offline_comparison.json'
with open(offline_path, 'w') as f:
    json.dump(offline_results, f, indent=2)
print(f'\nOffline results saved to {offline_path}')

---
## 8. Constrained Cache Comparison

Compare at reduced cache capacity (4 experts instead of 8+) to simulate
memory-constrained scenarios where policy quality matters more.

In [ ]:
from moe_policylang.compiler import compile_policy
from moe_policylang.dsl import MoEPolicyLang

# Build constrained policies (capacity=4, half the experts)
sched = MoEPolicyLang()

@sched.policy
def constrained_lru(p):
    p.cache(capacity=4, eviction='lru')
    p.schedule(mode='gpu_only')

@sched.policy
def constrained_lfu_prefetch(p):
    p.cache(capacity=4, eviction='lfu', lfu_decay=0.9)
    p.prefetch(strategy='history', budget=2, history_window=60)
    p.schedule(mode='hybrid', cpu_threshold_ms=40.0)

@sched.policy
def constrained_score(p):
    p.cache(capacity=4, eviction='score', score_ema_alpha=0.3)
    p.prefetch(strategy='affinity', budget=2, affinity_threshold=0.3)
    p.schedule(mode='hybrid', cpu_threshold_ms=40.0)

print('=== Constrained Cache (capacity=4) on Mixtral trace ===')
constrained_results = {}

for name, ir in sched.policies.items():
    compiled = compile_policy(ir)
    result = harness.run_policy(compiled, workload)
    m = result.metrics
    constrained_results[name] = {
        'hit_rate': m.hit_rate,
        'evictions': m.evictions,
        'us_per_layer': m.mean_dispatch_us,
    }
    print(f'  {name:<30} hit={m.hit_rate:.4f} evict={m.evictions:>5} lat={m.mean_dispatch_us:.1f}us')

# Also run baseline at capacity=4
from moe_policylang.baselines import HandCodedLRU
result = harness.run_baseline(HandCodedLRU, workload, capacity=4, baseline_name='baseline_lru_cap4')
m = result.metrics
constrained_results['baseline_lru_cap4'] = {
    'hit_rate': m.hit_rate,
    'evictions': m.evictions,
    'us_per_layer': m.mean_dispatch_us,
}
print(f'  {"baseline_lru_cap4":<30} hit={m.hit_rate:.4f} evict={m.evictions:>5} lat={m.mean_dispatch_us:.1f}us')

---
## Notes for Paper

- **Table 3 / Figure X**: Use `results` dict for throughput comparison
- **Table 4**: Use `offline_results` for dispatch overhead percentages
- **Section 5.3**: Use `constrained_results` to show policy quality differences under pressure
- All measurements use n=5 runs with bootstrap 95% CIs
- VRAM usage is peak `torch.cuda.max_memory_allocated()`